# Demo 1 — Silver Transformation

This notebook transforms both Bronze tables into clean Silver tables.

It creates:

- `demo1_crypto_ohlcv_silver`
- `demo1_crypto_ticks_silver`

The Silver layer:

- removes duplicate records
- keeps only valid rows
- standardizes field names and data types
- adds useful derived columns
- separates clean analytics-ready data from raw Bronze metadata


## 1. Load shared configuration

In [0]:
%run ../config/00_config


## 2. Import required Spark functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## 3. Confirm both Bronze tables exist

In [0]:
required_bronze_tables = [
    historical_bronze_table,
    streaming_bronze_table,
]

missing_bronze_tables = [
    table_name
    for table_name in required_bronze_tables
    if not spark.catalog.tableExists(table_name)
]

if missing_bronze_tables:
    raise RuntimeError(
        f"Missing Bronze tables: {missing_bronze_tables}"
    )

print("Both Bronze tables are available.")


# Part A — Historical OHLCV Silver

## 4. Read the historical Bronze table

In [0]:
historical_bronze_df = spark.table(
    historical_bronze_table
)

print(
    f"Historical Bronze rows: "
    f"{historical_bronze_df.count()}"
)


## 5. Clean and enrich historical OHLCV data

This transformation:

- keeps valid OHLC rows
- removes duplicate `symbol + open_time` records
- calculates daily price change
- calculates daily percentage change
- calculates candle range
- adds a bullish/bearish candle direction


In [0]:
historical_window = (
    Window
    .partitionBy(
        "symbol",
        "open_time",
    )
    .orderBy(
        F.col("ingested_at").desc()
    )
)

historical_silver_df = (
    historical_bronze_df
    .filter(
        F.col("symbol").isNotNull()
        & F.col("open_time").isNotNull()
        & F.col("open").isNotNull()
        & F.col("high").isNotNull()
        & F.col("low").isNotNull()
        & F.col("close").isNotNull()
        & (F.col("high") >= F.col("low"))
        & (F.col("volume") >= 0)
    )
    .withColumn(
        "_row_number",
        F.row_number().over(
            historical_window
        ),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
    .withColumn(
        "trade_date",
        F.to_date("open_time"),
    )
    .withColumn(
        "price_change",
        F.col("close") - F.col("open"),
    )
    .withColumn(
        "price_change_pct",
        F.when(
            F.col("open") != 0,
            (
                (
                    F.col("close")
                    - F.col("open")
                )
                / F.col("open")
            )
            * 100,
        ),
    )
    .withColumn(
        "candle_range",
        F.col("high") - F.col("low"),
    )
    .withColumn(
        "candle_direction",
        F.when(
            F.col("close") > F.col("open"),
            F.lit("BULLISH"),
        )
        .when(
            F.col("close") < F.col("open"),
            F.lit("BEARISH"),
        )
        .otherwise(
            F.lit("NEUTRAL")
        ),
    )
    .select(
        "record_id",
        "symbol",
        "interval",
        "trade_date",
        "open_time",
        "close_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "price_change",
        "price_change_pct",
        "candle_range",
        "candle_direction",
        "source_system",
        "source_file_name",
        "ingested_at",
    )
)

display(
    historical_silver_df
    .orderBy(
        "symbol",
        "open_time",
    )
)


## 6. Write the historical Silver table

`overwriteSchema = true` keeps the notebook easy to rerun during development.


In [0]:
(
    historical_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        historical_silver_table
    )
)

print(
    f"Historical Silver table written: "
    f"{historical_silver_table}"
)


## 7. Validate historical Silver

In [0]:
historical_silver_result_df = spark.table(
    historical_silver_table
)

historical_silver_summary_df = (
    historical_silver_result_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("row_count"),
        F.min("trade_date").alias(
            "first_trade_date"
        ),
        F.max("trade_date").alias(
            "last_trade_date"
        ),
        F.avg("price_change_pct").alias(
            "average_daily_change_pct"
        ),
    )
    .orderBy("symbol")
)

display(historical_silver_summary_df)


# Part B — Streaming Ticks Silver

## 8. Read the streaming Bronze table

In [0]:
streaming_bronze_df = spark.table(
    streaming_bronze_table
)

print(
    f"Streaming Bronze rows: "
    f"{streaming_bronze_df.count()}"
)


## 9. Clean and deduplicate streaming events

The Silver transformation:

- removes invalid events
- keeps one row per `event_id`
- keeps one row for duplicated Event Hub partition-offset pairs
- adds event date and event minute
- calculates ingestion delay
- preserves the evolved 24-hour market fields


In [0]:
event_id_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("ingested_at").desc()
    )
)

partition_offset_window = (
    Window
    .partitionBy(
        "eventhub_partition",
        "eventhub_offset",
    )
    .orderBy(
        F.col("ingested_at").desc()
    )
)

streaming_silver_df = (
    streaming_bronze_df
    .filter(
        F.col("event_id").isNotNull()
        & F.col("symbol").isNotNull()
        & F.col("price_usd").isNotNull()
        & (F.col("price_usd") > 0)
        & F.col("event_time").isNotNull()
    )
    .withColumn(
        "_event_id_row_number",
        F.row_number().over(
            event_id_window
        ),
    )
    .filter(
        F.col("_event_id_row_number") == 1
    )
    .drop("_event_id_row_number")
    .withColumn(
        "_offset_row_number",
        F.row_number().over(
            partition_offset_window
        ),
    )
    .filter(
        F.col("_offset_row_number") == 1
    )
    .drop("_offset_row_number")
    .withColumn(
        "event_date",
        F.to_date("event_time"),
    )
    .withColumn(
        "event_minute",
        F.date_trunc(
            "minute",
            F.col("event_time"),
        ),
    )
    .withColumn(
        "ingestion_delay_seconds",
        (
            F.col("ingested_at").cast("long")
            - F.col("event_time").cast("long")
        ),
    )
    .select(
        "event_id",
        "symbol",
        "price_usd",
        "event_time",
        "event_date",
        "event_minute",
        "change_pct_24h",
        "high_price_24h",
        "low_price_24h",
        "volume_24h",
        "producer_id",
        "source_system",
        "eventhub_partition",
        "eventhub_offset",
        "eventhub_enqueued_at",
        "ingested_at",
        "ingestion_delay_seconds",
    )
)

display(
    streaming_silver_df
    .orderBy(
        F.col("event_time").desc()
    )
)


## 10. Write the streaming Silver table

In [0]:
(
    streaming_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        streaming_silver_table
    )
)

print(
    f"Streaming Silver table written: "
    f"{streaming_silver_table}"
)


## 11. Validate streaming Silver

In [0]:
streaming_silver_result_df = spark.table(
    streaming_silver_table
)

streaming_silver_summary_df = (
    streaming_silver_result_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias(
            "first_event_time"
        ),
        F.max("event_time").alias(
            "last_event_time"
        ),
        F.min("price_usd").alias(
            "minimum_price"
        ),
        F.max("price_usd").alias(
            "maximum_price"
        ),
        F.avg(
            "ingestion_delay_seconds"
        ).alias(
            "average_ingestion_delay_seconds"
        ),
    )
    .orderBy("symbol")
)

display(streaming_silver_summary_df)


## 12. Final Silver validation

In [0]:
historical_silver_rows = (
    historical_silver_result_df.count()
)

streaming_silver_rows = (
    streaming_silver_result_df.count()
)

historical_duplicate_count = (
    historical_silver_result_df
    .groupBy(
        "symbol",
        "open_time",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

streaming_duplicate_event_count = (
    streaming_silver_result_df
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print("Silver transformation summary")
print("-----------------------------")
print(
    f"Historical Silver rows: "
    f"{historical_silver_rows}"
)
print(
    f"Streaming Silver rows: "
    f"{streaming_silver_rows}"
)
print(
    f"Historical duplicate keys: "
    f"{historical_duplicate_count}"
)
print(
    f"Streaming duplicate event IDs: "
    f"{streaming_duplicate_event_count}"
)

if historical_duplicate_count > 0:
    raise RuntimeError(
        "Historical Silver contains duplicate keys."
    )

if streaming_duplicate_event_count > 0:
    raise RuntimeError(
        "Streaming Silver contains duplicate event IDs."
    )

print("Silver transformations completed successfully.")


## Board explanation

> Bronze keeps source-oriented data and ingestion metadata. Silver applies quality rules, removes duplicates, standardizes records, and adds reusable analytical fields. Historical and streaming data remain in separate Silver tables because they represent different granularities: daily candles and live price events.

## Next notebook

`transformation/09_gold_aggregation.ipynb`
